# Project Mobility — Architecture
**Catalog:** `project_mobility` | **Platform:** Databricks Community Edition | **Storage:** Delta Lake

## Medallion Architecture Overview

| Layer | Schema | Purpose |
|---|---|---|
| Bronze | `bronze` | Raw ingested data — no transformations, audit columns only |
| Silver | `silver` | Cleaned, standardized, deduplicated |
| Gold | `gold` | Aggregated, model-ready and BI-ready |

## Data Sources & Ingestion

| Source | Format | Ingestion Method | Bronze Table |
|---|---|---|---|
| NYC TLC Green Taxi | Parquet (HTTP) | `requests` + `spark.createDataFrame` | `bronze.taxi_trips_raw` |
| NOAA Weather | CSV (Volume) | `spark.read.csv` | `bronze.weather_raw` |
| Public Holidays & Events | CSV (Volume) | `spark.read.csv` | `bronze.holidays_raw` |
| EIA Fuel Prices | CSV (Volume) | `spark.read.csv` | `bronze.fuel_prices_raw` |
| NYC Taxi Zone Lookup | CSV (HTTP) | `requests` + `spark.createDataFrame` | `bronze.taxi_zone_lookup` |

## End to End Data Flow

```
SOURCES                BRONZE                        SILVER                      GOLD                    CONSUMERS
───────                ──────                        ──────                      ────                    ─────────

NYC TLC (HTTP)  ──►  bronze.taxi_trips_raw       ──► silver.taxi_clean        ──►                      
NOAA CSV (Volume)   ──►  bronze.weather_raw          ──► silver.weather_clean     ──► gold.demand_forecast_features ──► Model 1
Manual CSV (Volume) ──►  bronze.holidays_raw         ──► silver.holidays_clean    ──► gold.fare_prediction_features ──► Model 2
EIA CSV (Volume)    ──►  bronze.fuel_prices_raw      ──► silver.fuel_prices_clean ──► gold.tip_prediction_features  ──► Model 3
TLC CSV (HTTP)  ──►  bronze.taxi_zone_lookup     ──► silver.taxi_zone_enriched──► gold.taxi_demand_summary      ──► Power BI
```

## Bronze Layer — Raw Tables

### `bronze.taxi_trips_raw`
- **Source:** NYC TLC — Green Taxi Trip Records
- **Frequency:** Monthly (2025-01 to 2025-12)
- **Partition:** `year`, `month`
- **Audit Columns:** `source_file`, `ingestion_timestamp`

### `bronze.weather_raw`
- **Source:** NOAA Climate Data — Central Park Station (USW00094728)
- **Frequency:** Daily
- **Key Fields:** date, temperature_max, temperature_min, precipitation, snow

### `bronze.holidays_raw`
- **Source:** Manual CSV uploaded to S3
- **Key Fields:** date, event_name, event_type, expected_impact

### `bronze.fuel_prices_raw`
- **Source:** EIA Weekly Retail Gasoline Prices — NYC Region
- **Frequency:** Weekly
- **Key Fields:** week_start_date, price_per_gallon

### `bronze.taxi_zone_lookup`
- **Source:** NYC TLC Zone Lookup CSV
- **Key Fields:** location_id, borough, zone, service_zone

## Silver Layer — Cleaned Tables

| Table | From | Transformations |
|---|---|---|
| `silver.taxi_clean` | `bronze.taxi_trips_raw` | Remove nulls, cast types, filter outliers, add zone names |
| `silver.weather_clean` | `bronze.weather_raw` | Standardise units, fill missing dates |
| `silver.holidays_clean` | `bronze.holidays_raw` | Normalise event types, add impact flags |
| `silver.fuel_prices_clean` | `bronze.fuel_prices_raw` | Align weekly prices to daily dates |
| `silver.taxi_zone_enriched` | `bronze.taxi_zone_lookup` | Enrich with borough metadata |

## Gold Layer — Model & BI Ready Tables

| Table | Feeds | Description |
|---|---|---|
| `gold.demand_forecast_features` | Model 1 | Trips per zone/hour joined with weather, holidays, fuel |
| `gold.fare_prediction_features` | Model 2 | Trip-level features joined with weather and zones |
| `gold.tip_prediction_features` | Model 3 | Trip-level features for tip classification |
| `gold.taxi_demand_summary` | Power BI | Aggregated demand KPIs by zone, borough, time |

## ML Models

---

### Model 1 — Demand Forecasting
> **Question:** How many trips will happen in a zone next hour?

| Input Feature | Source Table |
|---|---|
| Trip counts per zone/hour | `silver.taxi_clean` |
| Temperature, rain, snow | `silver.weather_clean` |
| Is holiday, event type | `silver.holidays_clean` |
| Weekly fuel cost | `silver.fuel_prices_clean` |
| Borough, neighborhood | `silver.taxi_zone_enriched` |

**Output:** Predicted trip count per zone per hour  
**Algorithm:** Regression / Time Series

---

### Model 2 — Fare Prediction
> **Question:** Given a trip, what will the fare be?

| Input Feature | Source Table |
|---|---|
| Distance, duration, passenger count | `silver.taxi_clean` |
| Rain/snow (affects trip duration) | `silver.weather_clean` |
| Pickup/dropoff borough | `silver.taxi_zone_enriched` |

**Output:** Predicted fare amount  
**Algorithm:** Regression

---

### Model 3 — Tip Prediction
> **Question:** Will this passenger tip, and how much?

| Input Feature | Source Table |
|---|---|
| Fare amount, payment type, time of day | `silver.taxi_clean` |
| Trip distance | `silver.taxi_clean` |
| Pickup zone wealth indicator | `silver.taxi_zone_enriched` |

**Output:** Tip amount or no tip  
**Algorithm:** Classification